In [17]:
from __future__ import annotations

import io
import fitz
from PIL import Image
from pathlib import Path
from pprint import pprint
from collections.abc import Mapping
from IPython.display import display

from statistics import median
from dataclasses import dataclass

from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

In [2]:
import json
from pathlib import Path
from docling.document_converter import DocumentConverter
from docling.datamodel.document import DoclingDocument
from docling_core.types.doc.document import PictureItem, TableItem, ListItem

In [3]:
def load_docling_document(pdf_path: str | Path) -> DoclingDocument:
    """
    Load a DoclingDocument from cache if available.
    Otherwise, convert the PDF and create the cache automatically.
    """

    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    # Auto-generate cache path
    cache_path = pdf_path.with_suffix(".docling.json")

    # Load from cache
    if cache_path.exists():
        print(f"✅ Cache found. Loading cached document: {cache_path}")
        doc = DoclingDocument.load_from_json(cache_path)
        print(f"✅ Cache loaded as DoclingDocument")

    # Convert PDF and create cache
    else:
        print("⚠️ Cache not found.")
        print("⏳ Converting PDF using Docling DocumentConverter...")

        converter = DocumentConverter()
        result = converter.convert(pdf_path)
        doc = result.document

        # Save cache for future reuse
        doc.save_as_json(cache_path)

        print(f"✅ Conversion complete.")
        print(f"💾 Cache saved to: {cache_path}")

    # -----------------------------
    # Print document metadata
    # -----------------------------
    print("\n📄 Document Metadata")
    print("-" * 40)

    try:
        print(f"File Name      : {pdf_path.name}")
        print(f"Total Pages    : {len(doc.pages)}")
        print(f"Total Tables   : {len(doc.tables)}")
        print(f"Total Pictures : {len(doc.pictures)}")

        # Optional additional stats
        if hasattr(doc, "texts"):
            print(f"Text Blocks    : {len(doc.texts)}")

    except Exception as e:
        print(f"⚠️ Could not extract full metadata: {e}")

    print("-" * 40)

    return doc

In [6]:
# Example usage
food_pdf_path = (
    "/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/"
    "iid-platform/sample_data/named/"
    "govt-food-data-report-large-table-heavy.pdf"
)
aviation_pdf_path = (
    "/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/"
    "iid-platform/sample_data/named/"
    "cruising-heights-march-2026.pdf"
)
food_doc = load_docling_document(food_pdf_path)
food_fitz_pdf = fitz.open(food_pdf_path)
aviation_doc = load_docling_document(aviation_pdf_path)
aviation_fitz_pdf = fitz.open(aviation_pdf_path)

✅ Cache found. Loading cached document: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/iid-platform/sample_data/named/govt-food-data-report-large-table-heavy.docling.json
✅ Cache loaded as DoclingDocument

📄 Document Metadata
----------------------------------------
File Name      : govt-food-data-report-large-table-heavy.pdf
Total Pages    : 585
Total Tables   : 474
Total Pictures : 455
Text Blocks    : 2748
----------------------------------------
⚠️ Cache not found.
⏳ Converting PDF using Docling DocumentConverter...


[INFO] 2026-05-27 12:37:41,421 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 12:37:41,615 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 12:37:41,616 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 12:37:41,773 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 12:37:41,848 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-27 12:37:41,848 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer

✅ Conversion complete.
💾 Cache saved to: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/iid-platform/sample_data/named/cruising-heights-march-2026.docling.json

📄 Document Metadata
----------------------------------------
File Name      : cruising-heights-march-2026.pdf
Total Pages    : 64
Total Tables   : 1
Total Pictures : 142
Text Blocks    : 2135
----------------------------------------


In [11]:
table0 = aviation_doc.tables[0]
df0 = table0.export_to_dataframe()
# df0.head()
df0

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,Route,Pre-conflict rate,March 2026
0,South Asia ¦ Europe,~$2.50/kg,~$4.37/kg
1,South Asia ¦ North America,~$4/kg,~$6/kg
2,India export spot rates,350-400/kg,"up to ` 1,600/kg"


In [18]:
# type(aviation_doc.tables)
t_table = aviation_doc.tables[0]
# d = dict(t_table)
# pprint(d)
def r(x):
    if isinstance(x, Mapping):
        return {k: r(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [r(v) for v in x]
    try:
        return r(dict(x))
    except:
        return x

# pprint(r(t_table))

In [55]:
table = food_doc.tables[80]
df = table.export_to_dataframe()
df

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,E,FRUITS,,,,,,,,,
0,E001,"Apple, big (Malus domestica)",6,1.46±0.51,0.15±0.04,,,,,0.15±0.04,3.65±0.47
1,E002,"Apple, green (Malus domestica)",6,2.45±0.40,0.10±0.04,,,,,0.10±0.04,2.13±0.09
2,E003,"Apple, small (Malus domestica)",6,1.86±0.19,0.07±0.01,,,,,0.07±0.01,2.18±0.40
3,E004,"Apple, small, Kashmir (Malus domestica)",1,2.04,0.05,,,,,0.05,2.55
4,E005,"Apricot, dried (Prunus armeniaca)",6,3.98±0.06,0.07±0.01,0.07±0.02,0.29±0.04,0.09±0.01,,0.11±0.01,5.17±1.17
5,E006,"Apricot, processed (Prunus armeniaca)",3,4.31±0.14,0.01±0.01,0.01±0.01,0.07±0.00,0.01±0.00,,0.01±0.01,6.14±0.76
6,E007,Avocado fruit ( ) Persea sp.,1,2.10,0.02,,,,71,0.02,38.74
7,E008,Bael fruit (Aegle marmelos),1,1.60,0.60,,,,,0.60,4.50
8,E009,"Banana, ripe, montham (Musa x paradisiaca)",1,0.20,0.08,0.07,,,,0.09,2.20
9,E010,"Banana, ripe, poovam (Musa x paradisiaca)",2,0.24,0.07,0.08,,,,0.08,2.60


In [58]:
# pprint(r(food_doc.tables[80]))
# food_doc.tables[80]
aviation_doc.tables[0]

TableItem(self_ref='#/tables/0', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TABLE: 'table'>, prov=[ProvenanceItem(page_no=48, bbox=BoundingBox(l=48.59783172607422, t=649.0498199462891, r=512.0574340820312, b=586.2693939208984, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))], source=[], comments=[], captions=[], references=[], footnotes=[], image=None, data=TableData(table_cells=[TableCell(bbox=BoundingBox(l=69.9931, t=149.08769999999993, r=93.9961, b=156.79623990147775, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=0, end_col_offset_idx=1, text='Route', column_header=True, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=221.7511, t=149.08769999999993, r=289.1251000000001, b=156.79623990147775, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1